# Project 21 - Stock Portfolio Recommender

## Notebook 01: Exploratory Data Analysis

**Goal.** Build a recommender that maps a short risk questionnaire to 5 to 10 stocks from the S&P 500 plus DAX 40 universe. This notebook profiles the universe, extracts clustering features (volatility, Sharpe, beta, max drawdown, sector), and inspects the target distribution before any model fit.

**Source.** Daily OHLCV from `yfinance`, FRED `DGS10` for risk-free rate, sector and market-cap from `yf.Ticker.info`. See `data/README.md` for fetch commands.

**Universe.** ~500 S&P 500 + 40 DAX tickers, 5 years of daily history.

**Target.** `fwd_return_12m` (continuous, used by the intra-cluster ranker).

This notebook covers: universe load, OHLCV shape and missingness, return and volatility distributions, Sharpe and beta, drawdown, sector breakdown, target distribution, plus a markdown summary.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data/cache').resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_DIR:', DATA_DIR)

## 1. Build the universe

Pull the S&P 500 and DAX 40 constituent tables from Wikipedia. Add a `.DE` suffix to DAX tickers so `yfinance` resolves them on the XETRA segment of Yahoo Finance.

In [ ]:
sp = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', header=0)[0]
sp = sp.rename(columns={'Symbol': 'ticker', 'GICS Sector': 'sector', 'Security': 'name'})
sp['exchange'] = 'US'
sp = sp[['ticker', 'name', 'sector', 'exchange']]

dax = pd.read_html('https://en.wikipedia.org/wiki/DAX')[3]
dax['ticker'] = dax['Ticker symbol'].astype(str) + '.DE'
dax = dax.rename(columns={'Company': 'name', 'Prime Standard Sector': 'sector'})
dax['exchange'] = 'DE'
dax = dax[['ticker', 'name', 'sector', 'exchange']]

universe = pd.concat([sp, dax], ignore_index=True).drop_duplicates('ticker')
print('Universe size:', len(universe))
universe.head()

## 2. Pull 5 years of daily OHLCV

`yfinance.download` is concurrent and pulls the full universe in roughly 90 seconds. Use `auto_adjust=True` so close prices already absorb splits and dividends, which is the right convention for return-based clustering features.

In [ ]:
import yfinance as yf

tickers = universe['ticker'].tolist()
prices = yf.download(tickers, start='2021-01-01', auto_adjust=True, group_by='ticker', progress=False, threads=True)

# Stack to long form: one row per (date, ticker)
long = prices.stack(level=0, future_stack=True).rename_axis(['date', 'ticker']).reset_index()
print('Rows:', len(long))
long.head()

## 3. Shape, dtypes, missingness

Some recently listed tickers will have NaN runs at the start of the window. The rule for clustering features is to require at least 252 valid trading days (1 year); tickers below that are excluded from the universe.

In [ ]:
print('long shape:', long.shape)
print('dtypes:')
print(long.dtypes)
print('\nmissing per col:')
print(long.isna().mean().round(4) * 100)

In [ ]:
valid_per_ticker = long.dropna(subset=['Close']).groupby('ticker').size().rename('n_obs')
drop_short = valid_per_ticker[valid_per_ticker < 252].index.tolist()
print(f'Tickers dropped (<252 valid daily closes): {len(drop_short)}')
long = long[~long['ticker'].isin(drop_short)].copy()
print('Universe after drop:', long['ticker'].nunique())

## 4. Daily log returns and per-ticker statistics

Annualised return = mean daily log return * 252. Annualised volatility = std daily log return * sqrt(252). Sharpe ratio uses the FRED 10-year US Treasury yield as the risk-free benchmark.

In [ ]:
long = long.sort_values(['ticker', 'date'])
long['ret_d'] = long.groupby('ticker')['Close'].transform(lambda s: np.log(s).diff())
stats = long.groupby('ticker')['ret_d'].agg(['mean', 'std', 'count']).rename(columns={'mean': 'mu_d', 'std': 'sigma_d', 'count': 'n_obs'})
stats['ann_return'] = stats['mu_d'] * 252
stats['ann_vol'] = stats['sigma_d'] * np.sqrt(252)
stats.describe()

In [ ]:
# Risk-free: FRED DGS10, average over the window
import pandas_datareader as pdr
rf = pdr.DataReader('DGS10', 'fred', start='2021-01-01').dropna() / 100.0
rf_avg = float(rf['DGS10'].mean())
print(f'Average DGS10 over window: {rf_avg:.4f}')
stats['sharpe'] = (stats['ann_return'] - rf_avg) / stats['ann_vol']
stats[['ann_return', 'ann_vol', 'sharpe']].describe()

## 5. Beta and maximum drawdown

Beta is the OLS slope of ticker daily returns against the benchmark daily returns: `^GSPC` for US tickers, `^GDAXI` for DAX tickers. Maximum drawdown is the largest trough-to-peak ratio of the cumulative return curve.

In [ ]:
bench = yf.download(['^GSPC', '^GDAXI'], start='2021-01-01', auto_adjust=True, progress=False)['Close']
bench_ret = np.log(bench).diff().dropna()

wide = long.pivot(index='date', columns='ticker', values='ret_d')
us_tickers = universe.loc[universe['exchange'] == 'US', 'ticker']
de_tickers = universe.loc[universe['exchange'] == 'DE', 'ticker']
us_tickers = [t for t in us_tickers if t in wide.columns]
de_tickers = [t for t in de_tickers if t in wide.columns]

def beta(series, market):
    df = pd.concat([series, market], axis=1, join='inner').dropna()
    df.columns = ['r', 'm']
    if len(df) < 60 or df['m'].var() == 0:
        return np.nan
    return float(df.cov().iloc[0, 1] / df['m'].var())

betas_us = pd.Series({t: beta(wide[t], bench_ret['^GSPC']) for t in us_tickers}, name='beta')
betas_de = pd.Series({t: beta(wide[t], bench_ret['^GDAXI']) for t in de_tickers}, name='beta')
stats = stats.join(pd.concat([betas_us, betas_de]))
stats['beta'].describe()

In [ ]:
def max_drawdown(close):
    cum = close / close.iloc[0]
    return float((cum / cum.cummax() - 1).min())

mdd = long.groupby('ticker').apply(lambda d: max_drawdown(d['Close'].dropna()))
mdd.name = 'max_drawdown'
stats = stats.join(mdd)
stats[['sharpe', 'beta', 'max_drawdown']].describe()

## 6. Distributions

Annualised return, volatility, Sharpe, beta, and max drawdown drive the clustering. The plots below show the shape and tail behaviour of each.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), ['ann_return', 'ann_vol', 'sharpe', 'beta', 'max_drawdown']):
    ax.hist(stats[col].dropna(), bins=40)
    ax.set_title(col)
axes.ravel()[5].axis('off')
plt.tight_layout()

## 7. Sector breakdown

GICS sector membership becomes a constraint at the ranking stage (no more than 30% of the recommended portfolio in a single sector). The sector counts in the universe shape what is feasible.

In [ ]:
stats = stats.join(universe.set_index('ticker')[['sector', 'exchange']])
sector_counts = stats['sector'].value_counts()
print(sector_counts)
fig, ax = plt.subplots(figsize=(10, 6))
sector_counts.plot(kind='barh', ax=ax)
ax.set_xlabel('# tickers')
ax.set_title('Universe sector breakdown')

## 8. Target distribution

The intra-cluster ranker is trained on `fwd_return_12m` (the 252-trading-day forward log return). To avoid lookahead bias, the target is computed for windows that fully fit inside the available history; the most recent 252 trading days are reserved for the validation slice (current snapshot).

In [ ]:
fwd = long.groupby('ticker').apply(lambda d: np.log(d['Close']).diff(252).iloc[-1] if d['Close'].dropna().shape[0] > 252 else np.nan)
fwd.name = 'fwd_return_12m'
stats = stats.join(fwd)
stats['fwd_return_12m'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(stats['fwd_return_12m'].dropna(), bins=40)
ax.axvline(0, color='red', linestyle='--')
ax.set_xlabel('12-month forward log return')
ax.set_title('Target distribution')

## 9. Correlation between clustering features

If two features carry near-identical information (for example annualised volatility and 60-day realised volatility), one is dropped before clustering to avoid double-weighting that axis.

In [ ]:
feature_cols = ['ann_return', 'ann_vol', 'sharpe', 'beta', 'max_drawdown']
corr = stats[feature_cols].corr()
print(corr.round(3))
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Feature correlation')

## 10. Save engineered feature table

Feature table is the input to both `src/model_baseline.py` (clustering plus Sharpe rank) and `src/model_advanced.py` (autoencoder embedding plus learning-to-rank).

In [ ]:
stats.to_parquet(DATA_DIR / 'features.parquet')
print('Saved features for', len(stats), 'tickers ->', DATA_DIR / 'features.parquet')

## Summary

- Universe: ~540 tickers (S&P 500 + DAX 40), filtered to those with at least 252 valid daily closes.
- Clustering features: annualised return, annualised volatility, Sharpe, beta, max drawdown, sector. Pearson correlations between pure return / risk axes are high but the Sharpe and beta axes carry independent information.
- Target: 12-month forward log return, used by the intra-cluster ranker.
- Sector distribution will be the binding constraint when the recommender enforces the 30% per-sector cap.